# Aula 06 - Notebook: Lógica de Predicados e Quantificadores em Redes de Sensores
## SCADA-Core Automática — Grupo 04: Classificação e Seleção de Grãos por Visão Computacional

Neste notebook implementamos a avaliação de predicados unários e relacionais e os quantificadores de primeira ordem $\forall$ (*FORALL*) e $\exists$ (*EXISTS*) sobre a malha de instrumentos, atuadores e lotes de grãos da planta industrial.


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionários em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import List, Callable, Any, Dict

@dataclass
class SensorProcesso:
    tag: str
    setor: str
    tipo: str
    valor: float
    unidade: str
    limite_min: float
    limite_max: float
    falha_comunicacao: bool = False

    @property
    def alarme_ativo(self) -> bool:
        if self.falha_comunicacao:
            return True
        return self.valor < self.limite_min or self.valor > self.limite_max

@dataclass
class Grao:
    id_grao: int
    cor_ideal: bool
    tamanho_ideal: bool
    formato_ideal: bool
    dano: bool
    praga: bool
    impureza: bool
    massa_g: float

@dataclass
class Reservatorio:
    tag: str
    nome: str
    nivel_pct: float
    capacidade_kg: float

# Funções de Quantificação de Primeira Ordem (FOL)
def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    """Quantificador Universal (∀x ∈ D, P(x))"""
    return all(predicado(x) for x in dominio)

def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    """Quantificador Existencial (∃x ∈ D, P(x))"""
    return any(predicado(x) for x in dominio)

def COUNT_IF(dominio: List[Any], predicado: Callable[[Any], bool]) -> int:
    """Contador de elementos que satisfazem o predicado"""
    return sum(1 for x in dominio if predicado(x))

print("[OK] Estruturas de dados e operadores FOL inicializados com sucesso!")


## 1. Malha de Sensores e Varredura de Integridade ($\mathcal{S}_{	ext{crit}}$)
Varredura de conformidade universal ($
orall s \in \mathcal{S}_{	ext{crit}}, 	ext{Saudavel}(s)$) e detecção existencial de alarmes ($\exists s \in \mathcal{S}_{	ext{crit}}, 	ext{AlarmeAtivo}(s)$).


In [ ]:
# Definição dos instrumentos da Planta de Seleção de Grãos (ISA 5.1)
rede_sensores = [
    SensorProcesso('LIT-101', 'Recepção/Funil', 'NIVEL', 65.0, '%', 15.0, 95.0),
    SensorProcesso('ST-201',  'Tração Esteira', 'VELOCIDADE', 1.2, 'm/s', 0.8, 1.5),
    SensorProcesso('JI-201',  'Motor Esteira',  'CORRENTE', 4.2, 'A', 0.0, 6.0),
    SensorProcesso('WT-301',  'Pesagem',        'MASSA', 25.4, 'g', 0.0, 100.0),
    SensorProcesso('XS-401',  'Visão/Trigger',  'PRESENCA', 1.0, 'bin', 0.0, 1.0),
    SensorProcesso('KSA-401', 'Visão/Câmera',   'STATUS', 1.0, 'bin', 1.0, 1.0),
    SensorProcesso('PAL-601', 'Pneumática',     'PRESSAO', 6.2, 'bar', 5.0, 8.0),
    SensorProcesso('ZSH-601', 'Ejetor FY-603',  'POSICAO', 0.0, 'bin', 0.0, 1.0),
    SensorProcesso('LIT-703', 'Silo Rejeito C', 'NIVEL', 42.0, '%', 0.0, 90.0),
    SensorProcesso('XA-901',  'Segurança',      'EMERGENCIA', 0.0, 'bin', 0.0, 0.0),
]

tags_criticas = {'XA-901', 'JI-201', 'PAL-601', 'KSA-401', 'LIT-703'}
sensores_criticos = [s for s in rede_sensores if s.tag in tags_criticas]

# Avaliação dos Quantificadores em Operação Normal
todos_comunicando = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)
todos_criticos_ok = FORALL(sensores_criticos, lambda s: not s.alarme_ativo)
existe_alarme_critico = EXISTS(sensores_criticos, lambda s: s.alarme_ativo)

print(f"1. Todos Sensores Comunicando (FORALL): {todos_comunicando}")
print(f"2. Todos Sensores Críticos em Faixa Segura (FORALL): {todos_criticos_ok}")
print(f"3. Existe Alarme Crítico Ativo (EXISTS): {existe_alarme_critico}")

tabela_sensores = [
    {
        "Tag": s.tag,
        "Setor": s.setor,
        "Variável": s.tipo,
        "Valor Lido": f"{s.valor} {s.unidade}",
        "Faixa Segura": f"[{s.limite_min} a {s.limite_max}] {s.unidade}",
        "Alarme": "ATIVO" if s.alarme_ativo else "NORMAL"
    }
    for s in rede_sensores
]
print("\n=== STATUS DA REDE DE SENSORES (SCADA-CORE) ===")
print(formatar_tabela(tabela_sensores))

assert todos_comunicando is True
assert todos_criticos_ok is True
assert existe_alarme_critico is False


## 2. Varredura de Lotes de Grãos ($\mathcal{G}$) e Classificação por Visão Computacional
Verificação da **Partição Exaustiva e Disjunta** ($
orall g \in \mathcal{G}, (p_A \oplus p_B \oplus p_C) = 1$) e cálculo da taxa de rejeição com quantificadores.


In [ ]:
# Lote de Grãos inspecionados no ciclo atual
lote_graos = [
    Grao(1,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.025), # Cat A
    Grao(2,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.024), # Cat A
    Grao(3,  cor_ideal=False, tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.023), # Cat B
    Grao(4,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=True,  praga=False, impureza=False, massa_g=0.021), # Cat C (dano)
    Grao(5,  cor_ideal=True,  tamanho_ideal=False, formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.020), # Cat B
    Grao(6,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=True,  impureza=False, massa_g=0.022), # Cat C (praga)
    Grao(7,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.026), # Cat A
    Grao(8,  cor_ideal=False, tamanho_ideal=False, formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.019), # Cat B
    Grao(9,  cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=True,  massa_g=0.035), # Cat C (impureza)
    Grao(10, cor_ideal=True,  tamanho_ideal=True,  formato_ideal=True,  dano=False, praga=False, impureza=False, massa_g=0.025), # Cat A
]

# Predicados de Classificação
def Conforme(g: Grao) -> bool:
    """Categoria A: Cor, tamanho e formato ideais sem dano, praga ou impureza"""
    return g.cor_ideal and g.tamanho_ideal and g.formato_ideal and not (g.dano or g.praga or g.impureza)

def Defeituoso(g: Grao) -> bool:
    """Categoria C: Presença de dano, praga, impureza ou deformações severas"""
    return g.dano or g.praga or g.impureza

def Secundario(g: Grao) -> bool:
    """Categoria B: Não é aprovado pleno nem rejeitado por defeito grave"""
    return not Conforme(g) and not Defeituoso(g)

# Verificação da Partição Exaustiva e Disjunta (∀g ∈ G, exatamente uma categoria é verdadeira)
particao_disjunta = FORALL(lote_graos, lambda g: (int(Conforme(g)) + int(Secundario(g)) + int(Defeituoso(g))) == 1)

total_graos = len(lote_graos)
qtd_cat_a = COUNT_IF(lote_graos, Conforme)
qtd_cat_b = COUNT_IF(lote_graos, Secundario)
qtd_cat_c = COUNT_IF(lote_graos, Defeituoso)
taxa_rejeicao = qtd_cat_c / total_graos

print(f"1. Partição Exaustiva e Disjunta Válida (FORALL): {particao_disjunta}")
print(f"2. Total Inspecionado: {total_graos} grãos | Cat A: {qtd_cat_a} | Cat B: {qtd_cat_b} | Cat C: {qtd_cat_c}")
print(f"3. Taxa Instantânea de Rejeição: {taxa_rejeicao*100:.1f}%")

tabela_graos = [
    {
        "ID": g.id_grao,
        "Cor OK": g.cor_ideal,
        "Tam OK": g.tamanho_ideal,
        "Form OK": g.formato_ideal,
        "Dano": g.dano,
        "Praga": g.praga,
        "Impureza": g.impureza,
        "Classificação": "Categoria A" if Conforme(g) else ("Categoria B" if Secundario(g) else "Categoria C (Ejetar)")
    }
    for g in lote_graos
]
print("\n=== CLASSIFICAÇÃO INDIVIDUAL DO LOTE DE GRÃOS ===")
print(formatar_tabela(tabela_graos))

assert particao_disjunta is True
assert (qtd_cat_a + qtd_cat_b + qtd_cat_c) == total_graos


## 3. Simulação de Injeção de Falha Dinâmica e Permissivo Geral ($c_{	ext{PERM}}$)
Simulação de queda de pressão pneumática (`PAL-601 = 3.2 bar`), violando o quantificador universal de segurança e disparando o bloqueio geral de operação.


In [ ]:
# Injeção de Falha: Queda de pressão no compressor (PAL-601)
sensor_pal = next(s for s in rede_sensores if s.tag == 'PAL-601')
sensor_pal.valor = 3.2 # bar (Abaixo do limite de 5.0 bar)

# Reservatórios
reservatorios = [
    Reservatorio('SILO-A', 'Silo Produto Aprovado', 45.0, 5000.0),
    Reservatorio('SILO-B', 'Silo Produto Secundário', 30.0, 3000.0),
    Reservatorio('SILO-C', 'Silo Rejeitos C', 42.0, 2000.0),
    Reservatorio('FUN-101', 'Funil de Recepção', 65.0, 500.0)
]

# Recálculo das condições globais
saude_rede = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)
seguranca_critica = FORALL(sensores_criticos, lambda s: not s.alarme_ativo)
existe_emergencia = EXISTS(sensores_criticos, lambda s: s.alarme_ativo)
silos_capacidade_ok = FORALL(reservatorios, lambda r: r.nivel_pct < 90.0)

# Permissivo Geral de Marcha c_PERM
c_PERM = saude_rede and seguranca_critica and silos_capacidade_ok

print(f"=== RESULTADO DO SCAN APÓS INJEÇÃO DE FALHA (PAL-601 = 3.2 bar) ===")
print(f"1. Todos Sensores Críticos Saudáveis (FORALL): {seguranca_critica}")
print(f"2. Existe Alarme Crítico Ativo (EXISTS): {existe_emergencia}")
print(f"3. Permissivo Geral de Operação (c_PERM): {c_PERM} -> BLOQUEIO OPERACIONAL!")

assert seguranca_critica is False
assert existe_emergencia is True
assert c_PERM is False
print("\n[OK] Varredura FOL validou o bloqueio determinístico da planta com sucesso!")
